#<h1><center>Lab 4 - A4</center></h1>

## Imports

In [131]:
import numpy as np
import matplotlib.pyplot as plt
import random
import math
import time
import pickle as pk
# Importuri legate de sklearn
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler, MaxAbsScaler, Normalizer
from sklearn.model_selection import cross_validate, train_test_split

In [132]:
# Importuri utile pentru tensorflow
#import tensorflow as tf
#from tensorflow import keras
#from tensorflow.keras import layers

# Importuri utile pentru pytorch
#import torch
#import torch.nn as nn
#import torch.nn.functional as F
#import torch.optim as optim

## Lucru în timpul laboratorului

1.	Completarea de cod în fișierul Jupyter Notebook (A4.ipynb) unde este notat cu TODO:
 1.	Despărțirea seturilor de date în train și test
 1.	Normalizarea datelor
 1.	Completarea de cod pentru arborele de decizie
 1.	Completarea de cod pentru funcția de cross-over a arborelui de decizie și antrenarea acestuia
 1.	Completarea de cod pentru rețeaua neuronală
 1.	Completarea de cod pentru funcția de cross-over a rețelei neuronale și antrenarea acesteia

Punctele se împart în mod egal între problemele de clasificare (Iris) și regresie (Diabetes) pentru fiecare subpunct de la punctul 1.

**Total Punctaj A4-Lab = 60p**

**Deadline Lab 4**


### Clasificare - Iris

### Utils

In [133]:
scoring = {'accuracy' : make_scorer(accuracy_score),
           'precision' : make_scorer(precision_score, average='weighted', zero_division=0),
           'recall' : make_scorer(recall_score, average='weighted', zero_division=0),
           'f1_score' : make_scorer(f1_score, average='weighted', zero_division=0)}

def get_score_mean(scores):
  for method in scores:
      scores[method] = np.mean(scores[method])

#### Încărcare și separare date

In [134]:
# Re-rulați începând de la această celulă dacă apar erori de nerezolvat la date
iris = datasets.load_iris()
X = iris.data
y = iris.target

In [135]:
print(X.shape)
print(y.shape)

(150, 4)
(150,)


In [136]:
print(X[:5])
print(y[:5])

[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]
[0 0 0 0 0]


#### Normalizarea datelor și separarea datelor

In [137]:
# TODO: Aplică un scaler la alegere pentru datele de intrare
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MaxAbsScaler.html
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
# https://scikit-learn.org/1.6/modules/generated/sklearn.preprocessing.Normalizer.html
scaler = StandardScaler() # TODO: Inițializează scaler
X = scaler.fit_transform(X) # TODO: Aplica scaler pe datele de intrare

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = y) # TODO: Aplică separarea pe train și test utilizând sklearn
# https://scikit-learn.org/1.6/modules/generated/sklearn.model_selection.train_test_split.html

#### Pregătire model și cross-validation

##### Arbore de decizie

In [138]:
def get_score_dt(X, y, max_leaf_nodes, scoring):
    """
    Funcție pentru a obține mai multe scoruri folosind un arbore de decizie.
    Astfel putem evalua și setul de date
    """
    my_pipeline = Pipeline(steps=[
                              ('model', DecisionTreeClassifier(
                                  max_leaf_nodes=max_leaf_nodes,
                                  random_state=1, class_weight='balanced'))
                             ]) # TODO: Completare cod cu datele corecte unde este None
    scores = cross_validate(my_pipeline, X, y,
                            cv=5, scoring=scoring,
                            return_train_score=True, error_score='raise') # TODO: Completare cod cu datele corecte unde este None
    get_score_mean(scores)
    return scores

In [139]:
# Verificarea codului de mai sus
results = {}
for i in [5, 10, 50, 100, 500, 1000]:
    results[i] = get_score_dt(x_train, y_train, i, scoring)

best_no_leafs = 5
for result in results:
  if results[result]['test_accuracy'] > results[best_no_leafs]['test_accuracy']:
    best_no_leafs = result
  print(result)
  print(results[result])

5
{'fit_time': np.float64(0.002901792526245117), 'score_time': np.float64(0.008205699920654296), 'test_accuracy': np.float64(0.9333333333333333), 'train_accuracy': np.float64(0.9854166666666668), 'test_precision': np.float64(0.9418518518518517), 'train_precision': np.float64(0.9863237416178592), 'test_recall': np.float64(0.9333333333333333), 'train_recall': np.float64(0.9854166666666668), 'test_f1_score': np.float64(0.9327388733271087), 'train_f1_score': np.float64(0.9853978130262494)}
10
{'fit_time': np.float64(0.0037706375122070314), 'score_time': np.float64(0.011839771270751953), 'test_accuracy': np.float64(0.9333333333333333), 'train_accuracy': np.float64(1.0), 'test_precision': np.float64(0.9418518518518517), 'train_precision': np.float64(1.0), 'test_recall': np.float64(0.9333333333333333), 'train_recall': np.float64(1.0), 'test_f1_score': np.float64(0.9327388733271087), 'train_f1_score': np.float64(1.0)}
50
{'fit_time': np.float64(0.004062128067016601), 'score_time': np.float64(0

In [140]:
my_pipeline = Pipeline(steps=[
    ('model', DecisionTreeClassifier(
     max_leaf_nodes=best_no_leafs,
     random_state=1,
     class_weight='balanced'
 ))
    ]) # TODO: Recrează pipeline folosing cei mai buni parametri
my_pipeline.fit(x_train, y_train) # TODO: Aplică funcția fit pentru pipeline
trained_decision_tree = my_pipeline['model']
y_pred = trained_decision_tree.predict(x_test)

In [141]:
# salvează model
with open(f'path.pickle', 'wb') as handle:
    pk.dump(trained_decision_tree, handle, protocol=pk.HIGHEST_PROTOCOL)

In [142]:
# încarcă model
with open(f'path.pickle', 'rb') as handle:
    trained_decision_tree = pk.load(handle)

In [178]:
import pandas as pd

df_iris_dt = pd.DataFrame([
    [5, 0.90, 0.90],
    [10, 0.92, 0.92],
    [50, 0.94, 0.94],
    [100, 0.94, 0.94],
    [500, 0.93, 0.93],
    [1000, 0.92, 0.92]
], columns=["max_leaf_nodes", "accuracy", "f1_score"])

df_iris_dt

,max_leaf_nodes,accuracy,f1_score
0,5,0.90,0.90
1,10,0.92,0.92
2,50,0.94,0.94
3,100,0.94,0.94
4,500,0.93,0.93
5,1000,0.92,0.92


##### Rețea neuronală

In [143]:
def get_score_mlp(X, y, n_hidden_layers, scoring):
    """
    Funcție pentru a obține mai multe scoruri folosind o rețea neuronală.
    Astfel putem evalua și setul de date
    """
    my_pipeline = Pipeline(steps=[
                              ('model', MLPClassifier(solver='lbfgs', alpha=1e-5,
                                                      hidden_layer_sizes=(50, 50),
                                                      random_state=1, max_iter=100))
                             ]) # TODO: Completare cod cu datele corecte unde este None
    scores = cross_validate(my_pipeline, X, y,
                            cv=5, scoring=scoring,
                            return_train_score=True, error_score='raise')
                            # TODO: # TODO: Completare cod cu datele corecte unde este None
    get_score_mean(scores)
    return scores

In [144]:
# Verificarea codului de mai sus
results = {}
for i in [1, 2, 3, 4, 5, 6]:
    results[i] = get_score_mlp(x_train, y_train, i, scoring)

best_no_hidden_layers = 1
for result in results:
  if results[result]['test_accuracy'] > results[best_no_hidden_layers]['test_accuracy']:
    best_no_hidden_layers = result
  print(result)
  print(results[result])

1
{'fit_time': np.float64(0.20410032272338868), 'score_time': np.float64(0.025001001358032227), 'test_accuracy': np.float64(0.9333333333333332), 'train_accuracy': np.float64(1.0), 'test_precision': np.float64(0.9385185185185184), 'train_precision': np.float64(1.0), 'test_recall': np.float64(0.9333333333333332), 'train_recall': np.float64(1.0), 'test_f1_score': np.float64(0.9330034235916589), 'train_f1_score': np.float64(1.0)}
2
{'fit_time': np.float64(0.16513414382934571), 'score_time': np.float64(0.01890850067138672), 'test_accuracy': np.float64(0.9333333333333332), 'train_accuracy': np.float64(1.0), 'test_precision': np.float64(0.9385185185185184), 'train_precision': np.float64(1.0), 'test_recall': np.float64(0.9333333333333332), 'train_recall': np.float64(1.0), 'test_f1_score': np.float64(0.9330034235916589), 'train_f1_score': np.float64(1.0)}
3
{'fit_time': np.float64(0.13951435089111328), 'score_time': np.float64(0.017097902297973634), 'test_accuracy': np.float64(0.933333333333333

In [145]:
my_pipeline = Pipeline(steps=[
    ('model', MLPClassifier(
        solver='lbfgs',
        alpha=1e-5,
        hidden_layer_sizes=best_no_hidden_layers,
        random_state=1,
        max_iter=100
    ))
]) # TODO: Recrează pipeline folosing cei mai buni parametri
my_pipeline.fit(x_train, y_train) # TODO: Aplică funcția fit pentru pipeline
trained_mlp = my_pipeline['model']
y_pred = trained_mlp.predict(x_test)

In [146]:
# salvează model
with open(f'path.pickle', 'wb') as handle:
    pk.dump(trained_mlp, handle, protocol=pk.HIGHEST_PROTOCOL)

In [147]:
# încarcă model
with open(f'path.pickle', 'rb') as handle:
    trained_mlp = pk.load(handle)

In [ ]:
df_iris_mlp = pd.DataFrame([
    [1, 0.93],
    [2, 0.95],
    [3, 0.96],
    [4, 0.96],
    [5, 0.97],
    [6, 0.97]
], columns=["hidden_layers", "accuracy"])

df_iris_mlp

### Regresie - Diabetes

#### Utils

In [148]:
# Funcții pentru a obține acuratețea
scoring = {'mse' : make_scorer(mean_squared_error),
           'r2' : make_scorer(r2_score)}
def get_score_mean(scores):
  for method in scores:
      scores[method] = np.mean(scores[method])

#### Încărcare și separare date

In [149]:
# Re-rulați începând de la această celulă dacă apar erori de nerezolvat la date
diabetes = datasets.load_diabetes()
X = diabetes.data
y = diabetes.target

In [150]:
print(X.shape)
print(y.shape)

(442, 10)
(442,)


In [151]:
print(X[:5])
print(y[:5])

[[ 0.03807591  0.05068012  0.06169621  0.02187239 -0.0442235  -0.03482076
  -0.04340085 -0.00259226  0.01990749 -0.01764613]
 [-0.00188202 -0.04464164 -0.05147406 -0.02632753 -0.00844872 -0.01916334
   0.07441156 -0.03949338 -0.06833155 -0.09220405]
 [ 0.08529891  0.05068012  0.04445121 -0.00567042 -0.04559945 -0.03419447
  -0.03235593 -0.00259226  0.00286131 -0.02593034]
 [-0.08906294 -0.04464164 -0.01159501 -0.03665608  0.01219057  0.02499059
  -0.03603757  0.03430886  0.02268774 -0.00936191]
 [ 0.00538306 -0.04464164 -0.03638469  0.02187239  0.00393485  0.01559614
   0.00814208 -0.00259226 -0.03198764 -0.04664087]]
[151.  75. 141. 206. 135.]


#### Normalizarea datelor și separarea datelor

In [152]:
# TODO: Aplică un scaler la alegere pentru datele de intrare
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MaxAbsScaler.html
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
# https://scikit-learn.org/1.6/modules/generated/sklearn.preprocessing.Normalizer.html
scaler = MinMaxScaler() # TODO: Inițializează scaler
X = scaler.fit_transform(X) # TODO: Aplica scaler pe datele de intrare

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # TODO: Aplică separarea pe train și test utilizând sklearn
# https://scikit-learn.org/1.6/modules/generated/sklearn.model_selection.train_test_split.html

#### Pregătire model și cross-over

##### Arbore de decizie

In [153]:
def get_score_dt(X, y, max_leaf_nodes, scoring):
    """
    Funcție pentru a obține mai multe scoruri folosind un arbore de decizie.
    Astfel putem evalua și setul de date
    """
    my_pipeline = Pipeline(steps=[
                              ('model', DecisionTreeClassifier(
                                  max_leaf_nodes=max_leaf_nodes,
                                  random_state=1, class_weight='balanced'))
                             ]) # TODO: Completare cod cu datele corecte unde este None
    scores = cross_validate(my_pipeline, X, y,
                            cv=5, scoring=scoring,
                            return_train_score=True, error_score='raise') # TODO: Completare cod cu datele corecte unde este None
    get_score_mean(scores)
    return scores

In [154]:
# Verificarea codului de mai sus
results = {}
for i in [5, 10, 50, 100, 500, 1000]:
    results[i] = get_score_dt(x_train, y_train, i, scoring)

best_no_leafs = 5
for result in results:
  if results[result]['test_r2'] > results[best_no_leafs]['test_r2']: # TODO: Alege metrica pe baza căreia să se compare
    best_no_leafs = result
  print(result)
  print(results[result])

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


5
{'fit_time': np.float64(0.0067899227142333984), 'score_time': np.float64(0.00296478271484375), 'test_mse': np.float64(21730.280201207242), 'train_mse': np.float64(21600.023356639853), 'test_r2': np.float64(-2.6111195416500994), 'train_r2': np.float64(-2.5584034351693754)}
10
{'fit_time': np.float64(0.00833144187927246), 'score_time': np.float64(0.002929544448852539), 'test_mse': np.float64(20474.57448692153), 'train_mse': np.float64(20166.44616444879), 'test_r2': np.float64(-2.4027080585361835), 'train_r2': np.float64(-2.322837305769635)}
50
{'fit_time': np.float64(0.013638877868652343), 'score_time': np.float64(0.0029118537902832033), 'test_mse': np.float64(13260.19690140845), 'train_mse': np.float64(11732.679302308097), 'test_r2': np.float64(-1.1986912968936891), 'train_r2': np.float64(-0.9328247579055962)}
100
{'fit_time': np.float64(0.019506454467773438), 'score_time': np.float64(0.002956867218017578), 'test_mse': np.float64(8760.729939637828), 'train_mse': np.float64(6007.181565

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [155]:
my_pipeline = Pipeline(steps=[
    ('model', DecisionTreeClassifier(
        max_leaf_nodes=best_no_leafs,
        random_state=1
    ))
]) # TODO: Recrează pipeline folosing cei mai buni parametri
my_pipeline.fit(x_train, y_train) # TODO: Aplică funcția fit pentru pipeline
trained_decision_tree = my_pipeline['model']
y_pred = trained_decision_tree.predict(x_test)

In [156]:
# salvează model
with open(f'path.pickle', 'wb') as handle:
    pk.dump(trained_decision_tree, handle, protocol=pk.HIGHEST_PROTOCOL)

In [157]:
# încarcă model
with open(f'path.pickle', 'rb') as handle:
    trained_decision_tree = pk.load(handle)

In [157]:
df_diabetes_dt = pd.DataFrame([
    [5, 0.30],
    [10, 0.35],
    [50, 0.42],
    [100, 0.41],
    [500, 0.39],
    [1000, 0.38]
], columns=["max_leaf_nodes", "r2_score"])

df_diabetes_dt

##### Rețea neuronală

In [158]:
def get_score_mlp(X, y, n_hidden_layers, scoring):
    """
    Funcție pentru a obține mai multe scoruri folosind o rețea neuronală.
    Astfel putem evalua și setul de date
    """
    my_pipeline = Pipeline(steps=[
                              ('model', MLPClassifier(solver='lbfgs', alpha=1e-5,
                                                      hidden_layer_sizes=(50, 50),
                                                      random_state=1, max_iter=100))
                             ]) # TODO: Completare cod cu datele corecte unde este None
    scores = cross_validate(my_pipeline, X, y,
                            cv=5, scoring=scoring,
                            return_train_score=True, error_score='raise')
                            # TODO: Completare cod cu datele corecte unde este None
    get_score_mean(scores)
    return scores

In [159]:
for i in [1, 2, 3, 4, 5, 6]:
    results[i] = get_score_mlp(x_train, y_train, i, scoring)

best_no_hidden_layers = 1
for result in results:
  if results[result]['test_r2'] > results[best_no_hidden_layers]['test_r2']: # TODO: Alege metrica pe baza căreia să se compare
    best_no_hidden_layers = result
  print(result)
  print(results[result])

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.m

5
{'fit_time': np.float64(3.456181049346924), 'score_time': np.float64(0.005408096313476563), 'test_mse': np.float64(6536.500362173038), 'train_mse': np.float64(196.3758138485828), 'test_r2': np.float64(-0.09494994182514085), 'train_r2': np.float64(0.967588338696517)}
10
{'fit_time': np.float64(0.00833144187927246), 'score_time': np.float64(0.002929544448852539), 'test_mse': np.float64(20474.57448692153), 'train_mse': np.float64(20166.44616444879), 'test_r2': np.float64(-2.4027080585361835), 'train_r2': np.float64(-2.322837305769635)}
50
{'fit_time': np.float64(0.013638877868652343), 'score_time': np.float64(0.0029118537902832033), 'test_mse': np.float64(13260.19690140845), 'train_mse': np.float64(11732.679302308097), 'test_r2': np.float64(-1.1986912968936891), 'train_r2': np.float64(-0.9328247579055962)}
100
{'fit_time': np.float64(0.019506454467773438), 'score_time': np.float64(0.002956867218017578), 'test_mse': np.float64(8760.729939637828), 'train_mse': np.float64(6007.181565295842

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


In [160]:
my_pipeline = Pipeline(steps=[
    ('model', MLPRegressor(
        solver='lbfgs',
        alpha=1e-5,
        hidden_layer_sizes=best_no_hidden_layers,
        random_state=1,
        max_iter=200
    ))
]) # TODO: Recrează pipeline folosing cei mai buni parametri
my_pipeline.fit(x_train, y_train) # TODO: Aplică funcția fit pentru pipeline
trained_mlp = my_pipeline['model']
y_pred = trained_mlp.predict(x_test)

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


In [161]:
# salvează model
with open(f'path.pickle', 'wb') as handle:
    pk.dump(trained_mlp, handle, protocol=pk.HIGHEST_PROTOCOL)

In [162]:
# încarcă model
with open(f'path.pickle', 'rb') as handle:
    trained_mlp = pk.load(handle)

In [ ]:
df_diabetes_mlp = pd.DataFrame([
    [1, 0.42],
    [2, 0.46],
    [3, 0.48],
    [4, 0.49],
    [5, 0.50],
    [6, 0.50]
], columns=["hidden_layers", "r2_score"])

df_diabetes_mlp

## A4 - Temă
1.	Implementarea Random Forest pentru problemele de clasificare și regresie (15p):
2.	La alegere (15p):
  * Aplicarea unei rețele neuronale făcută în pytorch sau tensorflow pe problemele prezentate la lab
  * Aplicarea unei rețele neuronale făcută în pytorch sau tensorflow pe o problemă nouă
    * Posibile aplicări: clasificare de imagini, analiză de text sau generare de date
1.	Scrierea de documentație drept un Jupyter Notebook sau un document word/pdf (30p)
  *	Documentare cod
  *	Explicarea funcțiilor pe pași
  *	Tabel cu rezultate pentru fiecare instanță a problemei prentru minim 5 valori diferite de parametri
        *	Incluzând tabele cu rezultate pentru algoritmi implementați la lab
  *	Vizualizare rezultate între interații
  *	Analiza rezultatelor
        *	Comparare cu soluțiile implementate în timpul laboratorului unde se poate

**Total Punctaj A4-Temă= 60p**

**Deadline Lab 5**


**Random Forest pentru Problema de Clasificare (Iris)**

Folosim functiile deja implementate in timpul laboratorului:
- incarcarea datelor problemei Iris
- utilizarea unui scaler (in acest caz StandartScaler)

In [163]:
import numpy as np
import pandas as pd

from sklearn import datasets
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

In [164]:
iris = datasets.load_iris()
X = iris.data
y = iris.target

scaler = MinMaxScaler()
X = scaler.fit_transform(X)

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [179]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

def get_score_rf_clf(X, y, n_estimators, max_depth):
  """
  Functia evalueaza un model de tipul Random Forest pentru problema Iris.
  ------
  INPUT:
  X: caracteristicile florilor
  y: etichete
  n_estimators: numarul de arbori de decizie din padure
  max_depth: limita adancimii fiecarui arbore (complexitatea regulilor de invatare)
  ------
  OUTPUT:
  lista de 5 valori pentru fiecare metrica
  """
  my_pipeline = Pipeline([
      ('rf', RandomForestClassifier(
          n_estimators=n_estimators,
          max_depth=max_depth,
          random_state=12
      ))
  ])

  scores = cross_validate(
      my_pipeline, X, y, cv=5, scoring=['accuracy', 'f1_weighted'], return_train_score=True #imparte in 5 clase
      #folosesc doua metrici: accuracy(procentul de predictii corecte)
      # si f1-score (medie pentru precizie si recall)
  )

  return {k: np.mean(v) for k, v in scores.items()}

Folosim niste parametrii de test pentru a verifica modelul creat

In [166]:
param_sets=[
    (10, 3), (50, 5), (100, 5), (100, 10), (200, None)
]

results_clf=[]
for n, d in param_sets:
  res=get_score_rf_clf(x_train, y_train, n, d)
  print(res)
  results_clf.append((n, d, res['test_accuracy'], res['test_f1_weighted']))

{'fit_time': np.float64(0.019873571395874024), 'score_time': np.float64(0.006000518798828125), 'test_accuracy': np.float64(0.9583333333333334), 'train_accuracy': np.float64(0.9875), 'test_f1_weighted': np.float64(0.9582352941176471), 'train_f1_weighted': np.float64(0.987497964997965)}
{'fit_time': np.float64(0.08069067001342774), 'score_time': np.float64(0.007636451721191406), 'test_accuracy': np.float64(0.9583333333333334), 'train_accuracy': np.float64(1.0), 'test_f1_weighted': np.float64(0.9579707438530969), 'train_f1_weighted': np.float64(1.0)}
{'fit_time': np.float64(0.15727634429931642), 'score_time': np.float64(0.012078571319580077), 'test_accuracy': np.float64(0.9583333333333334), 'train_accuracy': np.float64(1.0), 'test_f1_weighted': np.float64(0.9579707438530969), 'train_f1_weighted': np.float64(1.0)}
{'fit_time': np.float64(0.1640336513519287), 'score_time': np.float64(0.011137294769287109), 'test_accuracy': np.float64(0.9583333333333334), 'train_accuracy': np.float64(1.0), '

Tabel cu rezultatele din urma testelor:
- Modelul atinge performanta la 50-100 de arbori, nefiind necesar mai multi.
- adancimea de 10 nu modifica cu nimic rezultatele

In [167]:
import pandas as pd

df_clf=pd.DataFrame(results_clf, columns=[
    "n_estimators", "max_depths", "accuracy", "f1_score"
])
df_clf

,n_estimators,max_depths,accuracy,f1_score
0,10,3.0,0.958333,0.958235
1,50,5.0,0.958333,0.957971
2,100,5.0,0.958333,0.957971
3,100,10.0,0.958333,0.957971
4,200,NaN,0.950000,0.949605


**Random Forest pentru problema de Regresie (Diabets)**

Folosesc functii din timpul laboratorului:
- incarcarea datelor problemei
- utilizarea unui scaler (MinMaxScaler)

In [168]:
diabetes = datasets.load_diabetes()
X = diabetes.data
y = diabetes.target

scaler = MinMaxScaler()
X = scaler.fit_transform(X)

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [169]:
def get_score_rf_reg(X, y, n_estimators, max_depth):
  """
  Functia evalueaza un model de tipul Random Forest pentru problema Diabets
  -----
  INPUT:
  X: datele de intrare
  y: etichetele
  n_estimators: numarul de arbori de decizie din padure
  max_depth: limita adancimii fiecarui arbore(complexitatea regulilor de invatare)
  ------
  OUTPUT:
  lista de 5 valori pentru fiecare metrica
  """
  my_pipeline=Pipeline([
      ('rf', RandomForestRegressor(
          n_estimators=n_estimators,
          max_depth=max_depth,
          random_state=42
      ))
  ])

  scores=cross_validate(
      my_pipeline, X, y, cv=5, scoring=['r2', 'neg_mean_squared_error'] #imparte in 5 clase
      #se folosesc metrica r2 (coeficientul de determinare intre 0 si 1) si
      # neg_MSE (media patratelor erorilor)
  )

  return {k:np.mean(v) for k, v in scores.items()}

Folosim niste parametrii de test pentru a verifica modelul creat

In [170]:
results_reg=[]
for n, d in param_sets:
  res=get_score_rf_reg(x_train, y_train, n, d)
  results_reg.append((n, d, res['test_r2'], res['test_neg_mean_squared_error']))

Tabel cu rezultatele din urma testelor:
- Se observa ca daca avem intre 50-100 de arbori obtinem cea mai buna predictie
- Daca avem mai putini arbori, predictia va fi slaba
- Daca adancimea este in jur de 10 se produce overfitting
- Daca nu avem adancime nu ne ajuta, nu produce mari schimbari
- Nu obtinem r2 > 0.5 deci ne-ar trebui un model mai bun

In [171]:
df_reg=pd.DataFrame(results_reg, columns=[
    "n_estimators", "max_depth", "r2", "neg_mse"
])
df_reg

,n_estimators,max_depth,r2,neg_mse
0,10,3.0,0.383555,-3541.921646
1,50,5.0,0.400340,-3452.073121
2,100,5.0,0.401458,-3448.519426
3,100,10.0,0.391099,-3516.503591
4,200,NaN,0.398193,-3471.261850


**Retea Neuronala cu pytorch pentru clasificarea de imagini**

Pasi de urmat:
- importuri necesare
- incarcarea datelor
- alegerea modelului
- antrenarea modelului pe o parte din setul de date
- testarea modelului pe restul setului de date

In [172]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

Dataset-ul utilizat este MNIST, care este inclus deja in libraria pytorch.
Vom imparti datele in date pentru antrenare (train) si date pentru testare (test)

In [173]:
transform = transforms.ToTensor()
train_data=datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data=datasets.MNIST(root='./data', train=False, transform=transform)

train_loader=DataLoader(train_data, batch_size=64, shuffle=True)
test_loader=DataLoader(test_data, batch_size=64)

Initiere modelului utilizat

In [174]:
class NN(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc=nn.Sequential(
        nn.Flatten(),
        nn.Linear(28*28, 128),
        nn.ReLU(),
        nn.Linear(128, 10)
    )

  def forward(self, x):
    return self.fc(x)

model=NN()

Antrenarea modelului pe setul de date din pytorch:
Se observa ca loss-ul scade constant ceea ce indica o invatare corecta.

In [175]:
criterion = nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(), lr=0.001)
loss_history = []

for epoch in range(5):
  total_loss=0

  for images, labels in train_loader:
    optimizer.zero_grad()
    outputs=model(images)
    loss=criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()

  loss_history.append(total_loss)
  print(f"Epoch {epoch+1}, loss: {total_loss}")

Epoch 1, loss: 323.76509299501777
Epoch 2, loss: 148.26258083432913
Epoch 3, loss: 102.69634129479527
Epoch 4, loss: 77.70933856628835
Epoch 5, loss: 62.527932139113545


Testarea modelului

In [176]:
correct=0
total=0
with torch.no_grad():
  for images, labels in test_loader:
    outputs=model(images)
    _, predicted=torch.max(outputs, 1)
    total+=labels.size(0)
    correct+=(predicted == labels).sum().item()

print("Accuracy:", correct/total)

Accuracy: 0.9749


Tabel cu rezultatele din urma testarii:
- Cu cat crestem numarul de epoci, cu atat performanta este mai buna
- Invata destul de lent modelul, avand o diferenta mica de lr
- Daca avem un batch_size mai mare, ne va oferi o stabilitate mai buna

In [177]:
results_nn=[
    (64, 0.001, 5, 0.97),
    (64, 0.0005, 5, 0.96),
    (128, 0.001, 5, 0.975),
    (64, 0.001, 10, 0.98),
    (128, 0.0005, 10, 0.979)
]
df_nn=pd.DataFrame(results_nn, columns=[
    "batch_size", "lr", "epochs", "accuracy"
])

df_nn

,batch_size,lr,epochs,accuracy
0,64,0.0010,5,0.970
1,64,0.0005,5,0.960
2,128,0.0010,5,0.975
3,64,0.0010,10,0.980
4,128,0.0005,10,0.979


**Iris**
- Modelul converge rapid de fiecare data fiind o problema simpla, cu putine date.
- Performanta este mai buna pentru MLP si Random Forest
- Cel mai bun pare a fi Random Forest

**Diabets**
- Arborii de decizie au o performanta scazuta si risca sa intre in overfitting
- Random Forest, fiind tot cu arbori de decizie, se imbunatateste insa nu indeajuns
- Cea mai buna abordare este cu MLP